# CLSS 2022 Replication — Python Side

Replicates the core empirical exercise from:

> Coulombe, Leroux, Stevanovic, Surprenant (2022).  
> 'How is Machine Learning Useful for Macroeconomic Forecasting?'  
> *Journal of Applied Econometrics*, 37(5), 920–964.

**Setup:**
- Dataset: FRED-MD (current vintage)
- Target: INDPRO (Industrial Production Index, tcode=5: log-difference × 100)
- Models: AR (BIC), KRR (RBF, K-fold), RF (K-fold)
- Horizons: h = 1, 3, 6, 12
- OOS window: 2000-01 onwards (expanding)

> **Network required**: the FRED-MD download cell fetches the current vintage.
> Run the first cell once and the data is cached locally.

In [ ]:
import numpy as np
import pandas as pd

from macrocast.data.fred_md import load_fred_md
from macrocast.data.schema import MacroFrame, MacroFrameMetadata, VariableMetadata
from macrocast.pipeline.components import (
    CVScheme,
    LossFunction,
    Nonlinearity,
    Regularization,
    Window,
)
from macrocast.pipeline.experiment import FeatureSpec, ForecastExperiment, ModelSpec
from macrocast.pipeline.models import KRRModel, RFModel
from macrocast.evaluation.metrics import msfe, relative_msfe
from macrocast.evaluation.decomposition import decompose_treatment_effects

## 1. Load FRED-MD

Requires internet on first run. Data is cached under `~/.macrocast/cache/`.

In [ ]:
# Load current FRED-MD vintage
# Returns (data_df, tcodes_series) where first row contains transformation codes
fred_data, tcodes = load_fred_md()

print(f"FRED-MD shape: {fred_data.shape}")
print(f"Date range: {fred_data.index[0]} — {fred_data.index[-1]}")
print(f"Variables: {fred_data.shape[1]}")

In [ ]:
# FRED variable group mapping (CLSS 2022 Table A1 conventions)
FRED_GROUPS = {
    "output_income":  ["RPI", "W875RX1", "INDPRO", "IPFPNSS", "IPFINAL", "IPCONGD",
                       "IPDCONGD", "IPNCONGD", "IPBUSEQ", "IPMAT", "IPDMAT", "IPNMAT",
                       "IPMANSICS", "IPB51222S", "IPFUELS", "CUMFNS"],
    "labor":          ["HWI", "HWIURATIO", "CLF16OV", "CE16OV", "UNRATE", "UEMPMEAN",
                       "UEMPLT5", "UEMP5TO14", "UEMP15OV", "UEMP15T26", "UEMP27OV",
                       "CLAIMSx", "PAYEMS", "USGOOD", "CES1021000001", "USCONS",
                       "MANEMP", "DMANEMP", "NDMANEMP", "SRVPRD", "USTPU", "USWTRADE",
                       "USTRADE", "USFIRE", "USGOVT", "CES0600000007", "AWOTMAN",
                       "AWHMAN", "CES0600000008", "CES2000000008", "CES3000000008"],
    "housing":        ["HOUST", "HOUSTNE", "HOUSTMW", "HOUSTS", "HOUSTW", "PERMIT",
                       "PERMITNE", "PERMITMW", "PERMITS", "PERMITW"],
    "prices":         ["WPSFD49207", "WPSFD49502", "WPSID61", "WPSID62", "OILPRICEx",
                       "PPICMM", "CPIAUCSL", "CPIAPPSL", "CPITRNSL", "CPIMEDSL",
                       "CUSR0000SAC", "CUSR0000SAD", "CUSR0000SAS", "CPIULFSL",
                       "CUSR0000SA0L2", "CUSR0000SA0L5", "PCEPI", "DDURRG3M086SBEA",
                       "DNDGRG3M086SBEA", "DSERRG3M086SBEA"],
    "money":          ["M1SL", "M2SL", "M2REAL", "AMBSL", "TOTRESNS", "NONBORRES",
                       "BUSLOANS", "REALLN", "NONREVSL", "CONSPI", "MZMSL",
                       "DTCOLNVHFNM", "DTCTHFNM", "INVEST"],
    "interest_rates": ["FEDFUNDS", "CP3Mx", "TB3MS", "TB6MS", "GS1", "GS5", "GS10",
                       "AAA", "BAA", "COMPAPFFx", "TB3SMFFM", "TB6SMFFM", "T1YFFM",
                       "T5YFFM", "T10YFFM", "AAAFFM", "BAAFFM", "TWEXMMTH", "EXSZUSx",
                       "EXJPUSx", "EXUSUKx", "EXCAUSx"],
    "stock_market":   ["SP500", "SPINDUS", "SPDIVYD", "S&P PE ratio", "S&P: indust",
                       "VXOCLSx"],
}

def assign_group(col: str) -> str:
    for g, members in FRED_GROUPS.items():
        if col in members:
            return g
    return "other"

print("Group mapping ready.")

## 2. Build MacroFrame

In [ ]:
TARGET = "INDPRO"
assert TARGET in fred_data.columns, f"{TARGET} not found in FRED-MD"

var_meta = [
    VariableMetadata(
        name=col,
        tcode=int(tcodes[col]),
        group=assign_group(col),
        is_target=(col == TARGET),
    )
    for col in fred_data.columns
]

meta = MacroFrameMetadata(
    frequency="M",
    vintage="current",
    source="FRED-MD",
    variables=var_meta,
)

mf = MacroFrame(data=fred_data, metadata=meta)
print(mf)

## 3. Model grid

We run three Python-side models. Linear regularized models (Ridge, LASSO, ARDI)
are handled by macrocastR on the R side (see `macrocastR/R/models.R`).

In [ ]:
HORIZONS = [1, 3, 6, 12]
OOS_START = "2000-01"  # first OOS evaluation date

# Determine min_train_size from the OOS start date
oos_start_idx = fred_data.index.get_loc(pd.Timestamp(OOS_START))
# Add a small buffer to ensure enough lags
min_train = oos_start_idx  # all observations before OOS start
print(f"OOS start index: {oos_start_idx}  ({OOS_START})")
print(f"OOS observations available: {len(fred_data) - oos_start_idx}")

In [ ]:
models = [
    # KRR with RBF kernel — nonlinear, data-rich (uses PCA factors), K-fold CV, L2 loss
    ModelSpec(
        model_cls=KRRModel,
        nonlinearity=Nonlinearity.KRR,
        regularization=Regularization.RIDGE,
        cv_scheme=CVScheme.kfold(k=5),
        loss_function=LossFunction.L2,
        model_id="krr__ridge__kfold5__l2",
        model_kwargs={"cv_folds": 5},
    ),
    # RF — nonlinear, data-rich (uses PCA factors), K-fold CV, L2 loss
    ModelSpec(
        model_cls=RFModel,
        nonlinearity=Nonlinearity.RANDOM_FOREST,
        regularization=Regularization.FACTORS,
        cv_scheme=CVScheme.kfold(k=5),
        loss_function=LossFunction.L2,
        model_id="rf__factors__kfold5__l2",
        model_kwargs={"cv_folds": 5, "n_estimators": 200},
    ),
]

feat_spec = FeatureSpec(
    n_factors=8,
    n_lags=4,
    use_factors=True,
)

print(f"Models: {[m.model_id for m in models]}")

## 4. Run experiment

This runs the full expanding-window pseudo-OOS evaluation.
With FRED-MD (~800 obs) and two models across 4 horizons, expect ~5–15 minutes on CPU.

In [ ]:
exp = ForecastExperiment(
    macro_frame=mf,
    target_col=TARGET,
    horizons=HORIZONS,
    model_specs=models,
    feature_spec=feat_spec,
    window=Window.EXPANDING,
    min_train_size=min_train,
    n_jobs=-1,  # use all cores
)

result_set = exp.run()
df = result_set.to_dataframe()
print(f"Result rows: {len(df)}")
df.head()

## 5. Relative MSFE table

Relative MSFE = model MSFE / AR MSFE.  Values < 1 indicate improvement over AR.

The AR(BIC) benchmark is estimated in `macrocastR`; here we use a simple AR(4)
constructed from the target column's own lags as a stand-in.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

def ar4_benchmark_forecasts(y_full: np.ndarray, min_train: int, h: int) -> np.ndarray:
    """Expanding-window AR(4) direct h-step forecasts."""
    n = len(y_full)
    preds = []
    for t in range(min_train + h, n + 1):
        y_tr = y_full[:t - h]
        # Build lag matrix for training
        p = 4
        if len(y_tr) <= p:
            preds.append(np.nan)
            continue
        rows = [y_tr[i:len(y_tr) - p + i] for i in range(p)]
        X_tr = np.column_stack(rows[::-1])  # (T-p, p)
        y_target = y_full[p + h - 1: t - 1]  # direct h-step target
        min_len = min(len(X_tr), len(y_target))
        if min_len < 5:
            preds.append(np.nan)
            continue
        X_tr, y_target = X_tr[:min_len], y_target[:min_len]
        coef = np.linalg.lstsq(X_tr, y_target, rcond=None)[0]
        x_test = y_tr[-p:][::-1]
        preds.append(float(x_test @ coef))
    return np.array(preds)

print("AR(4) benchmark function defined.")

In [ ]:
# Apply transformation (tcode=5: Δlog × 100) to get stationary INDPRO series
from macrocast.data.transforms import apply_tcode

indpro_raw = fred_data[TARGET]
indpro = apply_tcode(indpro_raw, tcode=5).dropna()

print(f"Stationary INDPRO: {indpro.index[0]} — {indpro.index[-1]}, n={len(indpro)}")

In [ ]:
rows = []
for h in HORIZONS:
    # AR(4) benchmark forecasts
    ar_hat = ar4_benchmark_forecasts(indpro.values, min_train, h)
    ar_true = indpro.values[min_train + h: min_train + h + len(ar_hat)]
    ar_hat = ar_hat[: len(ar_true)]
    valid = ~np.isnan(ar_hat)
    msfe_ar = msfe(ar_true[valid], ar_hat[valid])

    row = {"h": h, "AR(4)": 1.00}
    for mid in df["model_id"].unique():
        sub = df[(df["model_id"] == mid) & (df["horizon"] == h)]
        if len(sub) == 0:
            row[mid] = np.nan
            continue
        msfe_m = msfe(sub["y_true"].values, sub["y_hat"].values)
        row[mid] = msfe_m / msfe_ar if msfe_ar > 0 else np.nan
    rows.append(row)

rel_msfe_table = pd.DataFrame(rows).set_index("h")
print("\nRelative MSFE (model / AR):")
print(rel_msfe_table.round(3).to_string())

## 6. Cumulative SFE plot

In [ ]:
import matplotlib.pyplot as plt
from macrocast.evaluation.metrics import csfe

h = 1  # plot for h=1
fig, ax = plt.subplots(figsize=(10, 4))

# AR(4) benchmark CSFE
ar_hat = ar4_benchmark_forecasts(indpro.values, min_train, h)
ar_true = indpro.values[min_train + h: min_train + h + len(ar_hat)]
ar_hat_v = ar_hat[:len(ar_true)]
valid = ~np.isnan(ar_hat_v)
csfe_ar = csfe(ar_true[valid], ar_hat_v[valid])
ax.plot(csfe_ar, label="AR(4)", color="black", linestyle="--")

for mid in df["model_id"].unique():
    sub = df[(df["model_id"] == mid) & (df["horizon"] == h)]
    csfe_m = csfe(sub["y_true"].values, sub["y_hat"].values)
    ax.plot(csfe_m, label=mid)

ax.set_title(f"Cumulative SFE — INDPRO, h={h}")
ax.set_xlabel("OOS observation")
ax.set_ylabel("Cumulative squared forecast error")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Four-component decomposition (CLSS 2022, Eq. 11)

Decomposes average OOS-R² gains across models into four orthogonal components:
nonlinearity, data richness (regularization), CV scheme, and loss function.

In [ ]:
# Build a result_df that includes a benchmark row (AR(4)) so decomposition works.
# We add AR(4) h=1 results as the benchmark model.
BENCH_ID = "ar4__none__bic__l2"

bench_rows = []
for h in HORIZONS:
    ar_hat = ar4_benchmark_forecasts(indpro.values, min_train, h)
    n = min(len(ar_hat), len(indpro) - min_train - h)
    ar_true_v = indpro.values[min_train + h: min_train + h + n]
    ar_hat_v = ar_hat[:n]
    valid = ~np.isnan(ar_hat_v)
    bench_rows.append(pd.DataFrame({
        "model_id": BENCH_ID,
        "nonlinearity": "linear",
        "regularization": "none",
        "cv_scheme": "bic",
        "loss_function": "l2",
        "horizon": h,
        "y_true": ar_true_v[valid],
        "y_hat": ar_hat_v[valid],
    }))

bench_df = pd.concat(bench_rows, ignore_index=True)
decomp_df = pd.concat([bench_df, df], ignore_index=True)

result = decompose_treatment_effects(
    result_df=decomp_df,
    benchmark_model_id=BENCH_ID,
    horizon=1,
)

print("\nDecomposition (OOS-R² regression coefficients):")
print(result.summary_df.to_string(index=False))

## 8. Save results

In [ ]:
import pathlib

out_dir = pathlib.Path("results")
out_dir.mkdir(exist_ok=True)

result_set.to_parquet(out_dir / "clss2022_python.parquet")
rel_msfe_table.to_csv(out_dir / "relative_msfe_table.csv")
result.summary_df.to_csv(out_dir / "decomposition_h1.csv", index=False)

print("Results saved to examples/results/")